In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Dataset/Pizza_Sales.csv')
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [3]:
df.isnull().sum()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   pizza_id           48620 non-null  int64         
 1   order_id           48620 non-null  int64         
 2   pizza_name_id      48620 non-null  object        
 3   quantity           48620 non-null  int64         
 4   order_date         48620 non-null  datetime64[ns]
 5   order_time         48620 non-null  datetime64[ns]
 6   unit_price         48620 non-null  float64       
 7   total_price        48620 non-null  float64       
 8   pizza_size         48620 non-null  object        
 9   pizza_category     48620 non-null  object        
 10  pizza_ingredients  48620 non-null  object        
 11  pizza_name         48620 non-null  object        
dtypes: datetime64[ns](2), float64(2), int64(3), object(5)
memory usage: 4.5+ MB


### Cleaning

In [8]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')
df['order_time'] = pd.to_datetime(df['order_time'], format='%H.%M.%S')

### Feature Engineering

In [24]:
df['Order_Day'] = df['order_date'].dt.day_name()
df['Order_Month'] = df['order_date'].dt.month_name()
df['Order_Hour'] = df['order_time'].dt.hour

### KPIs

In [ ]:
'''
•	Total Revenue = Sum of total_price
•	Total Pizzas Sold = Sum of quantity
•	Total Orders = Count of unique order_id
•	Average Order Value (AOV) = Total Revenue ÷ Total Orders
•	Average Pizza per Order = Total Pizzas Sold ÷ Total Orders
'''

In [16]:
total_revenue = sum(df['total_price'])
total_pizza_sold = sum(df['quantity'])
# nunique -> number of uniuque records
total_orders = df['order_id'].nunique()
average_order_value = total_revenue / total_orders
average_pizza_per_order = total_pizza_sold / total_orders

In [22]:
print(f'Total Revenue: Rs.{total_revenue}')
print(f'Total Pizza Quantity Sold Till Date: {total_pizza_sold}')
print(f'Total Orders: {total_orders}')
# print(f'Average Pizza Order Value: {round(average_order_value, 2)}')
print(f'Average Pizza Order Value: Rs.{average_order_value:,.2f}')
print(f'Average Pizza Per Order: {average_pizza_per_order:,.2f}')

Total Revenue: Rs.817860.05
Total Pizza Quantity Sold Till Date: 49574
Total Orders: 21350
Average Pizza Order Value: Rs.38.31
Average Pizza Per Order: 2.32


### Analysis

#### Daily Trend - Sales by Day of the Week

In [26]:
weekdays = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['Order_Day'] = pd.Categorical(df['Order_Day'], categories=weekdays)

In [28]:
daily_trend = df.groupby('Order_Day', observed=False)['total_price'].sum().reset_index().rename(
    columns = {'total_price': 'Total_Price'}
)
daily_trend['Total_Price'] = daily_trend['Total_Price'].map("Rs.{:,.2f}".format)
daily_trend

,Order_Day,Total_Price
0,Sunday,"Rs.102,116.45"
1,Monday,"Rs.110,471.60"
2,Tuesday,"Rs.115,594.45"
3,Wednesday,"Rs.116,731.20"
4,Thursday,"Rs.121,650.30"
5,Friday,"Rs.129,690.90"
6,Saturday,"Rs.121,605.15"


#### Hourly Trend - Sales by Hour of the Day

In [29]:
hourly_trend = df.groupby('Order_Hour')['total_price'].sum().reset_index().rename(
    columns = {'total_price': 'Total_Price'}
)
hourly_trend['Total_Price'] = hourly_trend['Total_Price'].map("Rs.{:,.2f}".format)
hourly_trend

,Order_Hour,Total_Price
0,9,Rs.83.00
1,10,Rs.303.65
2,11,"Rs.44,935.80"
3,12,"Rs.111,877.90"
4,13,"Rs.106,065.70"
5,14,"Rs.59,201.40"
6,15,"Rs.52,992.30"
7,16,"Rs.70,055.40"
8,17,"Rs.86,237.45"
9,18,"Rs.89,296.85"


#### Monthly Trend - Monthly Revenue and Orders

In [31]:
df['Order_Month_Num'] = df['order_date'].dt.month

In [43]:
monthly_trend = df.groupby(['Order_Month_Num', 'Order_Month']).agg(
    Total_Price = ('total_price', 'sum'),
    Total_Orders = ('order_id', 'nunique')
).reset_index().sort_values(ascending=False, by='Total_Price')

monthly_trend['Total_Price'] = monthly_trend['Total_Price'].map("Rs.{:,.2f}".format)
monthly_trend[['Order_Month', 'Total_Price', 'Total_Orders']]

,Order_Month,Total_Price,Total_Orders
0,January,"Rs.71,620.15",1929
2,March,"Rs.71,301.40",1864
10,November,"Rs.71,004.85",1844
6,July,"Rs.70,880.65",1860
3,April,"Rs.70,312.00",1829
7,August,"Rs.69,497.30",1835
5,June,"Rs.68,161.45",1771
9,October,"Rs.68,152.20",1782
4,May,"Rs.67,648.80",1765
1,February,"Rs.64,419.45",1648
